### Importing Libraries

In [1]:

import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import warnings

ModuleNotFoundError: No module named 'xgboost'

### Load and Prepare the Dataset

In [2]:
warnings.filterwarnings("ignore")

# Load data
train = pd.read_csv("train.csv")
val = pd.read_csv("val.csv")
test = pd.read_csv("test.csv")
# Prepare features and target
feature_cols = [f"p{i}" for i in range(1, 43)] + ["turn"]
target_col = "label_move_col"

X_train = train[feature_cols].values.astype(np.float32)
y_train = train[target_col].values

X_val = val[feature_cols].values.astype(np.float32)
y_val = val[target_col].values

X_test = test[feature_cols].values.astype(np.float32)

print(X_train.shape, X_val.shape, X_test.shape)

NameError: name 'warnings' is not defined

#### Train and  Evaluate XGBoost model

In [3]:
# Train XGBoost model
print("Training Tuned XGBoost...")

xgb = XGBClassifier(
    n_estimators=600,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.85,
    colsample_bytree=0.85,
    objective="multi:softprob",
    num_class=7,
    tree_method="hist",
    eval_metric="mlogloss",
    early_stopping_rounds=30,
    random_state=42
)

xgb.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

# Evaluate XGBoost on validation and training sets
pred_xgb_val = np.argmax(xgb.predict_proba(X_val), axis=1)
acc_xgb_val = accuracy_score(y_val, pred_xgb_val)

pred_xgb_train = np.argmax(xgb.predict_proba(X_train), axis=1)
acc_xgb_train = accuracy_score(y_train, pred_xgb_train)

print(f"XGBoost Train Accuracy: {acc_xgb_train:.4f}")
print(f"XGBoost Val Accuracy: {acc_xgb_val:.4f}")

Training Tuned XGBoost...


NameError: name 'XGBClassifier' is not defined

### Train and  Evaluate LightGBM model

In [4]:
# Train tuned LightGBM model
print("Training Tuned LightGBM...")

lgb = LGBMClassifier(
    objective="multiclass",
    num_class=7,
    learning_rate=0.03,
    n_estimators=1200,
    subsample=0.9,
    colsample_bytree=0.9,
    num_leaves=64,
    max_depth=-1,
    random_state=42
)

lgb.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    eval_metric="multi_logloss"
)

# Evaluate LightGBM on validation and training sets
pred_lgb_val = np.argmax(lgb.predict_proba(X_val), axis=1)
acc_lgb_val = accuracy_score(y_val, pred_lgb_val)

pred_lgb_train = np.argmax(lgb.predict_proba(X_train), axis=1)
acc_lgb_train = accuracy_score(y_train, pred_lgb_train)

print(f"LightGBM Train Accuracy: {acc_lgb_train:.4f}")
print(f"LightGBM Val Accuracy: {acc_lgb_val:.4f}")


Training Tuned LightGBM...


NameError: name 'LGBMClassifier' is not defined

#### Build hybrid model (weighted ensemble)

In [5]:
print("Building Hybrid Model...")

val_proba = (0.4 * xgb.predict_proba(X_val) + 0.6 * lgb.predict_proba(X_val))
val_pred = np.argmax(val_proba, axis=1)

acc_ens = accuracy_score(y_val, val_pred)
print(f"Hybrid Model Val Accuracy: {acc_ens:.4f}")

Building Hybrid Model...


NameError: name 'xgb' is not defined

In [ ]:

# Generate and save confusion matrices
print("Generating confusion matrices...")

# Training confusion matrix (using hybrid model's logic isn't applicable, so we use one model or ensemble on train)
# Since ensemble was only defined on val, we create hybrid-like prediction on train for consistency:
train_proba = (0.4 * xgb.predict_proba(X_train) + 0.6 * lgb.predict_proba(X_train))
train_pred = np.argmax(train_proba, axis=1)

cm_train = confusion_matrix(y_train, train_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_train, annot=True, fmt='d', cmap='Blues', xticklabels=range(7), yticklabels=range(7))
plt.title('Confusion Matrix - Training Data')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.savefig('confusion_matrix_train.png', dpi=300, bbox_inches='tight')
plt.close()

# Validation confusion matrix (final ensemble predictions)
cm_val = confusion_matrix(y_val, val_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_val, annot=True, fmt='d', cmap='Blues', xticklabels=range(7), yticklabels=range(7))
plt.title('Confusion Matrix - Validation Data')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.savefig('confusion_matrix_val.png', dpi=300, bbox_inches='tight')
plt.close()

print("Confusion matrices saved as 'confusion_matrix_train.png' and 'confusion_matrix_val.png'")

# Predict on test set
print("Predicting Test Set...")

test_proba = (0.4 * xgb.predict_proba(X_test) + 0.6 * lgb.predict_proba(X_test))
test_pred = np.argmax(test_proba, axis=1)

submission = pd.DataFrame({
    "id": test["id"],
    "label_move_col": test_pred
})

submission.to_csv("submission.csv", index=False)
print("submission.csv saved!")

Generating confusion matrices...


NameError: name 'xgb' is not defined

In [ ]:
# predict on test data
y_test_pred = model.predict(X_test)
y_test_pred = np.argmax(y_test_pred , axis=1).astype(int)
Prediction = pd.DataFrame({"id": range( 1 , len(y_test_pred) +1 ), "label_move_col" : y_test_pred })
Prediction.to_csv("submission.csv" , index = False)

NameError: name 'model' is not defined